<a href="https://colab.research.google.com/github/Ashoksai-tech/-Transformer-Based-Real-Time-Feedback-Insights/blob/main/RAG_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qU langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.2 MB/s eta 0:00:00


# Load the Document using Langchian Document Loader

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("/content/Ashok sai - Resume.pdf")

In [7]:
docs = loader.load()
docs[0].page_content[:500]

'Ashok Sai G\n9500557167 Chennai, India\naashoksai306@gmail.com github linkedin portfolio InterviewBit LeetCode\nSUMMARY\nMachine Learning Engineer with experience in Python, ML model development, and frameworks like TensorFlow.\nEager to solve real-world problems through AI, data preprocessing, and team collaboration. Skilled in testing and\ndeploying models with strong analytical abilities.\nEDUCATION\nJeppiaar Engineering College Chennai, India\nBachelor’s degree in Electronics and Communication Engine'

# Chunk the Document to reduce the size

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
textsplitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

In [14]:
docs = textsplitter.split_text(docs[0].page_content)

In [21]:
%pip install --upgrade --quiet  langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [42]:
import getpass
import os

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("you api key")

# Generate embeddings for chunked data

In [27]:
 from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [28]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [33]:
pip install FAISS-CPU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 40.7 MB/s eta 0:00:00


In [34]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_texts(docs, embeddings)

In [36]:
query = "What is the summary of the resume?"
docs_and_scores = db.similarity_search_with_score(query)

for doc, score in docs_and_scores:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content[:500]}...")

Score: 0.4538460969924927
Content: • Physician Notetaker - Medical Transcription and NLP Pipeline (April 2025) – Developed an AI-powered medical
transcription system using fine-tuned BioBERT and BART models for automated clinical documentation. Implemented NER
for extracting medical insights from physician-patient conversations. Built automated SOAP note generation. GitHub
• AI-Powered Scheme Research Application (Dec 2024) – Developed an AI-powered application using LangChain and
OpenAI APIs for NLP-driven insights. Implemented ...
Score: 0.4733494222164154
Content: • Automated Machine Learning and NLP Workflow Application (August 2024) – Developed an AI-powered workflow
automation tool that streamlines data preprocessing, model training, and evaluation. Integrated NLP-based insights
generation to support decision-making.. GitHub
• Face Recognition Based Attendance System(oct 2024) – Created an automated attendance system using Python and
OpenCV for face detection. Integrated with an 

# Set the prompt template for llm to get desired output from llm

In [37]:
from langchain.prompts import ChatPromptTemplate

template = """You are an assistant that answers questions about a resume.
Use the following context to answer the question:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# query from llm based on the relevant document

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema import StrOutputParser

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

# Create a retrieval chain
retrieval_chain = (
    {"context": db.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Example query
query = "What are the project titles listed in the resume?"

# Invoke the chain with the query
response = retrieval_chain.invoke(query)

print(response)

The resume lists the following project titles:

* Physician Notetaker - Medical Transcription and NLP Pipeline
* AI-Powered Scheme Research Application
* Customer Lifetime Value Prediction
* Automated Machine Learning and NLP Workflow Application
* Face Recognition Based Attendance System
* AI-Powered RAG Pipeline for YouTube Video QA
